# Notebook 7: Support Vector Machine (SVM) — Brain Tumor Classification

## Overview
Support Vector Machines find the **optimal hyperplane** that maximises the margin between classes.

### Key Concepts
* **Support vectors** – data points closest to the decision boundary.
* **Kernel trick** – maps data into a higher-dimensional space to make it linearly separable.
* **C parameter** – regularisation; smaller C → wider margin (more misclassification allowed).
* **Gamma** – kernel width; higher gamma → tighter fit.

### Kernels Demonstrated
| Kernel | Description |
|---|---|
| `linear` | Standard dot product |
| `poly` | Polynomial mapping |
| `rbf` | Radial Basis Function (Gaussian) |
| `sigmoid` | tanh kernel |

## Dataset — Brain Tumor Feature Dataset (Kaggle)
**Source:** https://www.kaggle.com/datasets/jakeshbohaju/brain-tumor

This tabular dataset contains image-derived features from brain MRI scans, with a binary label indicating whether a tumor is present.

### How to Download
```bash
pip install kaggle
# Place kaggle.json at ~/.kaggle/kaggle.json
kaggle datasets download -d jakeshbohaju/brain-tumor \
  -p data/brain_tumor --unzip
```

In [ ]:
# ─── Install (uncomment if needed) ─────────────────────────────────────────
# !pip install kaggle pandas scikit-learn matplotlib seaborn

# ─── Download dataset ───────────────────────────────────────────────────────
# !kaggle datasets download -d jakeshbohaju/brain-tumor \
#   -p data/brain_tumor --unzip

## 1. Imports and Data Loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, classification_report,
    ConfusionMatrixDisplay, roc_auc_score, roc_curve
)
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

# ── Load data ──────────────────────────────────────────────────────────────
TUMOR_PATHS = [
    'data/brain_tumor/Brain Tumor.csv',
    'data/brain_tumor/brain_tumor.csv',
    'data/brain_tumor/Brain_Tumor_Features.csv',
]

df = None
for path in TUMOR_PATHS:
    try:
        df = pd.read_csv(path)
        print(f"Kaggle dataset loaded from {path}: {df.shape}")
        break
    except FileNotFoundError:
        continue

if df is None:
    print("Kaggle file not found – creating synthetic brain tumor feature dataset.")
    np.random.seed(42)
    n = 3762   # match original dataset size

    # Simulate MRI-derived features (mean, variance, std, entropy, etc.)
    no_tumor   = np.random.randn(n // 2, 14) * 0.8 + 0
    has_tumor  = np.random.randn(n // 2, 14) * 1.2 + 1.5
    X_raw = np.vstack([no_tumor, has_tumor])
    y_raw = np.array([0] * (n // 2) + [1] * (n // 2))

    feature_names = [
        'Mean','Variance','Standard Deviation','Entropy','Skewness',
        'Kurtosis','Contrast','Energy','ASM','Homogeneity',
        'Dissimilarity','Correlation','Coarseness','Image'
    ]
    df = pd.DataFrame(X_raw, columns=feature_names)
    df['Class'] = y_raw
    print(f"Synthetic dataset created: {df.shape}")

df.head()

## 2. Exploratory Data Analysis

In [ ]:
print("Shape:", df.shape)
print("\nData types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isnull().sum().sum())

# Identify target column
TARGET = 'Class' if 'Class' in df.columns else df.columns[-1]
print(f"\nTarget column : {TARGET}")
print("Target distribution:")
print(df[TARGET].value_counts())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Class distribution
df[TARGET].value_counts().plot(kind='bar', ax=axes[0], color=['steelblue','salmon'])
axes[0].set_title('Class Distribution (0=No Tumor, 1=Tumor)')
axes[0].set_xlabel('Class'); axes[0].set_ylabel('Count')
axes[0].tick_params(rotation=0)

# Correlation heatmap (first 10 numeric features)
numeric_cols = df.select_dtypes(include=np.number).columns[:10]
sns.heatmap(df[numeric_cols].corr(), annot=False, cmap='coolwarm', ax=axes[1])
axes[1].set_title('Feature Correlation Heatmap')

plt.tight_layout()
plt.show()

## 3. Data Preprocessing

In [ ]:
# Keep only numeric columns
numeric_df = df.select_dtypes(include=np.number).copy()

# Fill any missing values with column median
numeric_df.fillna(numeric_df.median(), inplace=True)

X = numeric_df.drop(columns=[TARGET])
y = numeric_df[TARGET].astype(int)

print(f"Features: {X.shape}, Target: {y.shape}")

# Train / test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Feature scaling — critical for SVM
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"Train: {X_train_sc.shape}, Test: {X_test_sc.shape}")

## 4. SVM with Different Kernels

In [ ]:
kernels = ['linear', 'rbf', 'poly', 'sigmoid']
kernel_results = []

for kernel in kernels:
    svm = SVC(kernel=kernel, C=1.0, probability=True, random_state=42)
    svm.fit(X_train_sc, y_train)
    y_pred = svm.predict(X_test_sc)
    y_prob = svm.predict_proba(X_test_sc)[:, 1]
    kernel_results.append({
        'Kernel':   kernel,
        'Accuracy': accuracy_score(y_test, y_pred),
        'ROC-AUC':  roc_auc_score(y_test, y_prob)
    })
    print(f"Kernel={kernel:8s} | Accuracy={accuracy_score(y_test,y_pred):.4f} | AUC={roc_auc_score(y_test,y_prob):.4f}")

results_df = pd.DataFrame(kernel_results)
print("\nBest kernel by AUC:", results_df.loc[results_df['ROC-AUC'].idxmax(), 'Kernel'])

## 5. Best SVM Model — Detailed Evaluation

In [ ]:
best_kernel = results_df.loc[results_df['ROC-AUC'].idxmax(), 'Kernel']

best_svm = SVC(kernel=best_kernel, C=1.0, probability=True, random_state=42)
best_svm.fit(X_train_sc, y_train)
y_pred_best = best_svm.predict(X_test_sc)
y_prob_best = best_svm.predict_proba(X_test_sc)[:, 1]

print(f"Best Kernel    : {best_kernel}")
print(f"Test Accuracy  : {accuracy_score(y_test, y_pred_best):.4f}")
print(f"ROC-AUC        : {roc_auc_score(y_test, y_prob_best):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_best, target_names=['No Tumor','Tumor']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Confusion matrix
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_best,
    display_labels=['No Tumor', 'Tumor'],
    cmap='Blues', ax=axes[0]
)
axes[0].set_title(f'SVM ({best_kernel}) — Confusion Matrix')

# ROC curve comparison for all kernels
for kernel in kernels:
    svm_k = SVC(kernel=kernel, C=1.0, probability=True, random_state=42)
    svm_k.fit(X_train_sc, y_train)
    prob_k = svm_k.predict_proba(X_test_sc)[:, 1]
    fpr_k, tpr_k, _ = roc_curve(y_test, prob_k)
    auc_k = roc_auc_score(y_test, prob_k)
    axes[1].plot(fpr_k, tpr_k, lw=2, label=f'{kernel} (AUC={auc_k:.3f})')

axes[1].plot([0,1],[0,1],'k--', lw=1, label='Random')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curves — All SVM Kernels')
axes[1].legend(loc='lower right')

plt.tight_layout()
plt.show()

## 6. Hyperparameter Tuning (C and gamma for RBF kernel)

In [ ]:
param_grid = {
    'C':     [0.1, 1.0, 10.0, 100.0],
    'gamma': ['scale', 'auto', 0.001, 0.01, 0.1],
}

grid_cv = GridSearchCV(
    SVC(kernel='rbf', probability=True, random_state=42),
    param_grid,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=0
)
grid_cv.fit(X_train_sc, y_train)

print(f"Best params    : {grid_cv.best_params_}")
print(f"Best CV AUC    : {grid_cv.best_score_:.4f}")

tuned_svm  = grid_cv.best_estimator_
y_pred_t   = tuned_svm.predict(X_test_sc)
y_prob_t   = tuned_svm.predict_proba(X_test_sc)[:, 1]
print(f"Tuned Test Acc : {accuracy_score(y_test, y_pred_t):.4f}")
print(f"Tuned ROC-AUC  : {roc_auc_score(y_test, y_prob_t):.4f}")

## 7. Decision Boundary Visualisation (2D PCA projection)

In [ ]:
# Project to 2D for visualisation
pca = PCA(n_components=2, random_state=42)
X_train_2d = pca.fit_transform(X_train_sc)
X_test_2d  = pca.transform(X_test_sc)

svm_2d = SVC(kernel='rbf', C=1.0, probability=True, random_state=42)
svm_2d.fit(X_train_2d, y_train)

# Create mesh
h = 0.05
x_min, x_max = X_train_2d[:, 0].min() - 1, X_train_2d[:, 0].max() + 1
y_min, y_max = X_train_2d[:, 1].min() - 1, X_train_2d[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                     np.arange(y_min, y_max, h))
Z = svm_2d.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm')
for cls, colour, label in zip([0, 1], ['steelblue','salmon'], ['No Tumor','Tumor']):
    mask = y_test == cls
    plt.scatter(X_test_2d[mask, 0], X_test_2d[mask, 1],
                c=colour, label=label, alpha=0.6, edgecolors='k', s=30)
plt.xlabel('PCA Component 1')
plt.ylabel('PCA Component 2')
plt.title('SVM Decision Boundary (RBF kernel, 2D PCA projection)')
plt.legend()
plt.tight_layout()
plt.show()

## 8. Cross-Validation

In [ ]:
from sklearn.pipeline import Pipeline

svm_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm',    SVC(kernel='rbf', C=1.0, probability=True, random_state=42))
])

cv_scores = cross_val_score(svm_pipeline, X, y, cv=5, scoring='roc_auc', n_jobs=-1)

print("5-Fold CV ROC-AUC Scores:", np.round(cv_scores, 4))
print(f"Mean : {cv_scores.mean():.4f}")
print(f"Std  : {cv_scores.std():.4f}")

## Summary

* SVMs are powerful classifiers that work well on **high-dimensional, small-to-medium datasets**.
* **Feature scaling** is mandatory — SVM is not scale-invariant.
* The **RBF kernel** is the most commonly used and works well as a default.
* Tune `C` and `gamma` via cross-validation (GridSearchCV).
* For very large datasets, consider `LinearSVC` (much faster) or SGD-based alternatives.